# L9a: Linear Models for Classification
In this lecture, let's look at another use for linear models: __classification__, where we assign each example to one of a fixed set of categories (classes). We adapt a linear model for classification by transforming its continuous output into a class label.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Adapt a linear model for binary classification:__ A binary classifier assigns each example to one of two classes. We'll pass the output of a linear model through the sign function to get a class label, and see how the model parameters define a boundary between the two classes.
> * __Train a linear classifier with the Perceptron algorithm:__ The Perceptron learns the classifier parameters from labeled examples, and it changes them only when an example is misclassified or lies on the boundary. We'll work through the algorithm, show why each update helps, and see when it is guaranteed to separate the two classes.
> * __Evaluate a classifier with a confusion matrix:__ A confusion matrix counts the correct and incorrect predictions of each type. We'll compute accuracy, precision, recall, and specificity from these counts, and see why accuracy alone can mislead when one class is rare.

We'll demonstrate these ideas with an example, where we train a Perceptron to tell genuine banknotes from forged ones and evaluate its predictions on test data. Let's get started!
___

## Examples
We will use the following example to illustrate the key ideas of this lecture:

> [▶ Build a Perceptron classifier](CHEME-5800-L9a-Example-LinearModels-Classification-Perceptron-Fall-2026.ipynb). How well does a linear classifier do on data it has not seen? We split the UCI banknote dataset, four numerical features computed from a camera image of each of 1,372 genuine or forged banknotes, into training and testing sets, train the Perceptron on the training set, and classify the test banknotes. We count the mistakes and use a confusion matrix to see which kinds of errors the classifier makes.

___

## Binary Classification Problem
In a __binary classification__ problem, each example belongs to one of two classes, and we want to predict the class from the example's features. A linear model computes a continuous output (a score) from the features. We can transform the score using an __activation function__ $\sigma$, either directly into a class label or into the probability of each class.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-LinearlySeparable-Schematic.svg" width="480" style="max-width:100%; height:auto;" alt="Scatter plot with Coordinate 1 on the horizontal axis and Coordinate 2 on the vertical axis. Orange Class 1 points sit in the upper right and teal Class 2 points in the lower left. A dashed straight line labeled Separating Hyperplane (linear) runs diagonally between the two groups, with every orange point on one side and every teal point on the other."/>
</p>

__Linearly separable data.__ Each point is an example with two features (the coordinates), colored by its class. The dashed line puts every class 1 example on one side and every class 2 example on the other. Data that a straight boundary can split this way are called __linearly separable__.

Let's examine two binary classification strategies:

* [The Perceptron (Rosenblatt, 1957)](https://en.wikipedia.org/wiki/Perceptron) is an algorithm for binary classification that learns a linear decision boundary separating two classes. The Perceptron maps the score directly to a class label in $\{-1,+1\}$ using $\sigma(z) = \texttt{sign}(z)$.
* [Logistic regression](https://en.wikipedia.org/wiki/Logistic_regression) uses the [logistic function](https://en.wikipedia.org/wiki/Logistic_function), an S-shaped curve with values between 0 and 1, to transform the score into the probability that an example belongs to a class. We'll explore logistic regression later this week.

Today, we focus on __the Perceptron__.
___

## Perceptron
Suppose there exists a dataset $\mathcal{D} = \left\{(\mathbf{x}_{i},y_{i}) \mid i = 1,2,\dots,n\right\}$ with $n$ labeled examples, where $\mathbf{x}_{i}\in\mathbb{R}^{m}$ is an $m$-dimensional vector of features and $y_{i}\in\{-1,1\}$ is the class label of example $i$, e.g., assigned by an expert. In the figure above, class 1 could be $y=1$ and class 2 could be $y=-1$.

The Perceptron computes the estimated label $\hat{y}_{i}$ for example $i$ by passing the linear score through the $\texttt{sign}:\mathbb{R}\to\{-1,1\}$ function:
$$
\begin{equation*}
    \hat{y}_{i} = \texttt{sign}\left(\hat{\mathbf{x}}_{i}^{\top}\;\mathbf{\theta}\right)
\end{equation*}
$$
where the augmented features are $\hat{\mathbf{x}}_{i}^{\top}=\left(x_{i1},x_{i2},\dots,x_{im},1\right)$ (we've added a `1` to account for the bias term, so $p=m+1$), and $\mathbf{\theta}=\left(w_{1},\dots,w_{m}, b\right)\in\mathbb{R}^{p}$ is the vector of classifier parameters, with $w_{j}\in\mathbb{R}$ the weight of feature $j$ and $b\in\mathbb{R}$ the bias. The $\texttt{sign}(z)$ function is defined as:
$$
\begin{equation*}
    \texttt{sign}(z) =
    \begin{cases}
        1 & \text{if}~z\geq{0}\\
        -1 & \text{if}~z<0
    \end{cases}
\end{equation*}
$$

__Decision boundary.__ The set of feature vectors where the score $\hat{\mathbf{x}}^{\top}\mathbf{\theta} = w_{1}x_{1}+\dots+w_{m}x_{m}+b$ equals zero, $\left\{\mathbf{x}\in\mathbb{R}^{m} \mid \hat{\mathbf{x}}^{\top}\mathbf{\theta} = 0\right\}$, is the classifier's __decision boundary__. The score is positive on one side of the boundary and negative on the other. When at least one weight is nonzero, the boundary is a __hyperplane__: a line when $m=2$, as in the figure, and a plane when $m=3$. The weights set its orientation, and the bias shifts it away from the origin.

The product $y_{i}\,\hat{\mathbf{x}}_{i}^{\top}\mathbf{\theta}$ is positive when the score and the label have the same sign, i.e., when example $i$ is on the correct side of the boundary. We count example $i$ as a __mistake__ when $y_{i}\,\hat{\mathbf{x}}_{i}^{\top}\mathbf{\theta}\leq{0}$: it is on the wrong side of the boundary or on the boundary itself. The dataset $\mathcal{D}$ is __linearly separable__ if some parameter vector $\mathbf{\theta}$ makes this product positive for every example.

### Classical: Online Perceptron Training
The Perceptron learns $\mathbf{\theta}$ by passing through the training data repeatedly. It is an __online__ algorithm: it looks at one example at a time and updates the parameters right away whenever that example is a mistake.

Let's examine the Perceptron learning algorithm pseudocode.

__Initialize__: Given a dataset $\mathcal{D}$, the maximum number of passes through the data $T\geq{1}$, and a mistake threshold $M$ (we stop when a pass makes at most $M$ mistakes, e.g., $M=0$), initialize the parameter vector $\mathbf{\theta} = \left(w_{1},\dots,w_{m}, b\right)$ to zero or small random values and set the loop counter $t\gets{0}$.

While $\texttt{true}$ __do__:
1. Initialize the number of mistakes $\texttt{mistakes} \gets 0$.
2. For each training example $(\mathbf{x}, y) \in \mathcal{D}$, with augmented features $\hat{\mathbf{x}}$, __do__:
    - If $y\;\hat{\mathbf{x}}^{\top}\mathbf{\theta}\leq{0}$ __then__ the example is a __mistake__. Update $\mathbf{\theta} \gets \mathbf{\theta} + y\;\hat{\mathbf{x}}$ and increment $\texttt{mistakes} \gets \texttt{mistakes} + 1$.
3. After processing all examples, increment $t \gets t + 1$.
4. If $\texttt{mistakes} \leq {M}$ __or__ $t \geq T$ __then__ return $\mathbf{\theta}$. Otherwise, repeat from step 1.

__Why does this update help?__ Suppose the example $(\mathbf{x}, y)$ is a mistake, and let $\mathbf{\theta}^{\prime} = \mathbf{\theta} + y\,\hat{\mathbf{x}}$ be the updated parameters. Because $y^{2}=1$, the product of the label and the new score is given by:
$$
\begin{equation*}
    y\,\hat{\mathbf{x}}^{\top}\mathbf{\theta}^{\prime} = y\,\hat{\mathbf{x}}^{\top}\mathbf{\theta} + y^{2}\,\hat{\mathbf{x}}^{\top}\hat{\mathbf{x}} = y\,\hat{\mathbf{x}}^{\top}\mathbf{\theta} + \lVert\hat{\mathbf{x}}\rVert_{2}^{2}
\end{equation*}
$$
The product grows by $\lVert\hat{\mathbf{x}}\rVert_{2}^{2}$, the squared length of $\hat{\mathbf{x}}$, which is at least $1$ because the last entry of $\hat{\mathbf{x}}$ is $1$. Thus, the update moves this example's score toward the correct sign. One update may not be enough, and it also changes the scores of the other examples, so we pass through the data repeatedly.

__Convergence__: If the dataset $\mathcal{D}$ is linearly separable, $M=0$, and $T$ is large enough, the Perceptron finds a separating hyperplane after a finite number of updates. The following result bounds how many:

> __Perceptron convergence theorem (Novikoff, 1962):__ Suppose there is a parameter vector $\mathbf{\theta}^{\star}$ with $\lVert\mathbf{\theta}^{\star}\rVert_{2}=1$ and a margin $\gamma>0$ such that $y_{i}\,\hat{\mathbf{x}}_{i}^{\top}\mathbf{\theta}^{\star}\geq\gamma$ for every example in $\mathcal{D}$. Let $R = \max_{i}\lVert\hat{\mathbf{x}}_{i}\rVert_{2}$ be the length of the longest augmented feature vector. Then, starting from $\mathbf{\theta}=\mathbf{0}$, the number of updates $k$ the Perceptron makes satisfies:
>
> $$
> \begin{equation*}
>     k \leq \left(\frac{R}{\gamma}\right)^{2}
> \end{equation*}
> $$
>
> A larger margin $\gamma$ means $\mathbf{\theta}^{\star}$ separates the classes more cleanly. The bound depends on the ratio $R/\gamma$: well-separated classes need few updates, while classes that nearly touch can need many. It is an upper bound; the actual number of updates can be smaller.

A pass with zero mistakes leaves $\mathbf{\theta}$ unchanged, so the returned $\mathbf{\theta}$ classifies every training example correctly. If $\mathcal{D}$ is not linearly separable, every pass makes at least one mistake, so with $M=0$ the loop runs until it reaches the limit of $T$ passes. We rarely know in advance whether the data are separable, which is why the algorithm always has a limit $T$, and why we may accept a small number of mistakes $M>0$.

### Modern: Nonlinear Optimization
Later this week, we'll look at another way to train a linear classifier: we write a loss function that measures the classification error and minimize it with numerical optimization. The classical online Perceptron is a good starting point for understanding linear binary classification.
___

## Confusion Matrix for Binary Classification
A [confusion matrix](https://en.wikipedia.org/wiki/Confusion_matrix) evaluates binary classifier performance by comparing predicted labels to true (actual) labels. We call one class __positive__, usually the class we want to detect, and the other class __negative__. With labels $y\in\{-1,1\}$, the positive class is $y=1$.

> __Confusion Matrix Structure__
>
> The confusion matrix for a binary classifier is a $2\times{2}$ table that counts the examples in each combination of actual and predicted class:
>
>|                     | **Predicted Positive** | **Predicted Negative** |
>|---------------------|------------------------|------------------------|
>| **Actual Positive** | True Positive (TP)     | False Negative (FN)    |
>| **Actual Negative** | False Positive (FP)    | True Negative (TN)     |
>
> * **True Positive (TP)**: the number of positive examples that the classifier predicts as positive.
> * **False Negative (FN)**: the number of positive examples that the classifier predicts as negative. Each one misses a true case.
> * **False Positive (FP)**: the number of negative examples that the classifier predicts as positive. Each one is a false alarm.
> * **True Negative (TN)**: the number of negative examples that the classifier predicts as negative.
>
> The diagonal entries count the correct predictions, and the off-diagonal entries count the two kinds of mistakes.

In medical diagnosis, for example, the positive class consists of patients who have the disease. A false positive tells a healthy patient they have the disease, while a false negative fails to diagnose a patient who has it. The two kinds of mistakes can have very different costs, so we count them separately.

### Performance Metrics
The confusion matrix gives the counts we need to calculate four performance metrics:

* **Accuracy** $=\frac{TP + TN}{TP + TN + FP + FN}$: What fraction of all predictions were correct?
* **Precision** $=\frac{TP}{TP + FP}$: Of all positive predictions, what fraction were correct? High precision means few of the positive predictions are false alarms.
* **Recall (Sensitivity)** $=\frac{TP}{TP + FN}$: Of all actual positive examples, what fraction did the classifier identify? High recall means few of the positive examples are missed.
* **Specificity** $=\frac{TN}{TN + FP}$: Of all actual negative examples, what fraction did the classifier correctly identify? High specificity means few of the negative examples are false alarms.

> __Key insight__: Different applications require different trade-offs. In medical diagnosis, high recall (avoiding missed diagnoses) may be more important than high precision. When one class is rare, accuracy can mislead: if only 3% of the examples are positive, a classifier that predicts negative for every example is 97% accurate but has zero recall.

Let's examine an example applying the Perceptron algorithm and the confusion matrix.

> __Example__
>
> [▶ Build a Perceptron classifier](CHEME-5800-L9a-Example-LinearModels-Classification-Perceptron-Fall-2026.ipynb). Train the Perceptron on the banknote data, classify the test banknotes, and use a confusion matrix to break down the classifier's mistakes.

___

## Lab
In the L9b lab, we'll train the Perceptron on datasets that are and are not linearly separable, including the XOR (exclusive or) pattern, where the feature plane is cut into four wedges and diagonally opposite wedges share a class, and see where a linear classifier breaks down.

___

## Summary
In this lecture, we adapted linear models for binary classification, trained a linear classifier with the Perceptron algorithm, and evaluated classifiers with a confusion matrix.

> __Key Takeaways:__
>
> * __Linear models for classification:__ We passed the continuous output of a linear model through the sign function to get a class label. The parameters define a hyperplane, the decision boundary, and the classifier assigns one class to each side of it.
>
> * __Perceptron learning and convergence:__ We trained the Perceptron, which updates its parameters only when an example is misclassified or lies on the boundary, and showed that each update moves that example's score toward the correct sign. With a zero-mistake threshold and enough passes, it finds a separating hyperplane for linearly separable data after a finite number of updates, while on other data it never reaches zero mistakes, so a limit on the number of passes or a larger mistake threshold stops it.
>
> * __Performance evaluation:__ The confusion matrix counts the true positives, false positives, false negatives, and true negatives. From these counts, we defined accuracy, precision, recall, and specificity, and saw that accuracy alone can hide a classifier that misses every example of a rare class.

Later this week, we'll turn to logistic regression, which estimates the probability that an example belongs to each class, and to the numerical optimization methods we use to train it.

___